In [1]:
%load_ext autoreload
%autoreload 2

import os
import sys
from functools import partial

import ipywidgets as widgets
import jax
import jax.numpy as jnp
import matplotlib
import matplotlib.pylab as plt
import numpy as np
from IPython.core.display_functions import clear_output
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import MultiTaskLassoCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures
from tqdm import tqdm

# Add to PYTHONPATH the folders of the project so the scripts can be imported
this_file_path = os.getcwd()
project_name = "smooth-sciml"
chapter_name = "05-reduced-order-modeling"
chapter_path = os.path.abspath(os.path.join(this_file_path.split(chapter_name)[0], chapter_name))
chapter_src_path = os.path.abspath(os.path.join(chapter_path, "src"))
sys.path.insert(1, chapter_path)
sys.path.insert(1, chapter_src_path)
print(chapter_src_path)

from src.MakeFigures.example_of_transport import get_reconstruction, LINEAR_SVD, LINEAR_GREEDY, NCROM_RF, NCROM_NN, NCROM_Q
from src.lib.SKLearnLikeModels import SKJaxMLP
from src.lib.Utils4TransportEq import sample_space_for_non_linear_train, N_train_max

/home/callum/Repos/GitHub/smooth-sciml/chapters/05-reduced-order-modeling/src


## Non-linear reduced order model

Here we will use linear and non-linear reduced spaces to approximate the manifold of step functions
$$\mathcal{M} = \{u(\theta)(x) = \chi_{[0, \theta]} : \theta \in [0,1] \}.$$
For this we will use as linear reduced basis the Karhunen–Loève orthonormal basis of eigenfunctions $f_k$ which are in this case the trigonometric functions associated to the interval $[0, 1]$: $f_k^s = 2 \sin(2k\pi x)$ and $f_k^c = 2 \cos(2k\pi x)$.

For the non-linear strategies we use the strategy presented in (A. Cohen, et al. 2023) where the approximation is devided in a linear term and a non-linear approximation of the coefficients of the rest of the linear expansion:

$$
\widehat u_n(\theta) = \sum_{i=1}^n c_i(\theta) v_i + \sum_{i>n}^N \psi_i(c_1(\theta),\dots, c_n(\theta)) v_{i}.
$$

Here we choose $\psi$ to be
- A polynomial of degree 2: `Pipeline([("poly", PolynomialFeatures(degree=2)), ("lasso", MultiTaskLassoCV())])`
- A neural network of two layers with ten neurons each: `SKJaxMLP(layers=(10, 10), num_iters=10000, lr=1e-2)`
- A random forest regressor with ten sub-trees: `RandomForestRegressor(n_estimators=10)`

In the examples, to avoid recomputing at each change of parameters, you have to press on `Run Interact` to see the updated plot.

In [2]:
k_max = 100  # maximum number of Fourier terms for the approximations
Nx = 500  # number of discretization points in space
N_train_max = 10000
n_train = N_train_max  # number of examples to train

# Defines the parameter ranges of \theta, here is set to only vary the width being always of height=1 and left starting point in 0
c_lim = (0.0, 0.0, 1)
h_lim = (1.0, 1.0, 1)
w_lim = (0.0, 0.1, N_train_max)

TARGET = "Ground truth"
LINEAR_MODEL = "Linear model"
non_linear_models = {
    NCROM_RF: RandomForestRegressor(n_estimators=10),
    NCROM_NN: SKJaxMLP(layers=(10, 10), num_iters=10000, lr=1e-2),
    NCROM_Q: Pipeline([("poly", PolynomialFeatures(degree=2)), ("lasso", MultiTaskLassoCV())])
}
colors = matplotlib.colormaps['tab10'].colors
colors = matplotlib.colormaps['tab10'].colors
colors_dict = {
    TARGET: "black",
    r"$u(\theta)$": "black",
    LINEAR_MODEL: colors[0],
    LINEAR_SVD: colors[0],
    LINEAR_GREEDY: colors[1],
    NCROM_NN: colors[2],
    NCROM_RF: colors[3],
    NCROM_Q: colors[4]
}
    
errors = None
hashes = None
# global non_linear_models

### Explore reconstructions

In the following cell there is a widget to explore how the different strategies reconstruct the step-function of choice. You can play with the parameter $\theta$, the number of elements in the linear basis $n$ and which strategies to show between
- Linear SVD: linear ROM via the Karhunen–Loève basis.
- Linear Greedy: you can see that the approximations take the step functions that were in the basis.
- Non-linear via Neural Network (takes some seconds to train).
- non-linear via Random Forest (fast to train).
- Non-linear via a Quadratic model (can take long to train).

In [3]:
@widgets.interact_manual(
    # c=widgets.FloatSlider(min=c_lim[0], max=c_lim[1], step=0.05, value=np.min(c_lim)),
    # h=widgets.FloatSlider(min=h_lim[0], max=h_lim[1], step=0.05, value=np.mean(h_lim)),
    theta=widgets.FloatSlider(min=0, max=1, step=0.05, value=0.5),
    n_linear=widgets.IntSlider(min=1, max=k_max, step=2, value=3),
    linear_svd=False,
    linear_greedy=False,    
    non_linear_crom_nn=False,
    non_linear_crom_rf=False,
    non_linear_crom_q=False,
)
def reconstruction(theta, n_linear, linear_svd, linear_greedy, non_linear_crom_nn, non_linear_crom_rf, non_linear_crom_q):
    c=0.0 
    h=1.0
    
    models = []
    if linear_svd: models.append(LINEAR_SVD)
    if linear_greedy: models.append(LINEAR_GREEDY)
    if non_linear_crom_nn: models.append(NCROM_NN)
    if non_linear_crom_rf: models.append(NCROM_RF)
    if non_linear_crom_q: models.append(NCROM_Q)

    global non_linear_models
    global errors
    global hashes
    x, reconstruction, non_linear_models, errors, hashes = get_reconstruction(c, h, theta, n_linear, models, k_max, Nx, c_lim, h_lim, w_lim, n_train, non_linear_models, errors, hashes)
    clear_output(wait=True)
    fig, ax = plt.subplots(nrows=1, figsize=(8, 3))
    for label in reconstruction:
        ax.plot(x, reconstruction[label], label=label, c=colors_dict[label])
    ax.set_ylim((-0.2, h * 1.2))
    ax.legend()
    plt.show()

interactive(children=(FloatSlider(value=0.5, description='theta', max=1.0, step=0.05), IntSlider(value=3, desc…

### Explore the coefficient approximation error

In this case we can compare how the error that each method has in reconstructing the second term coefficients, that is
$$c_i \approx \psi_i(c_1(\theta),\dots, c_n(\theta))$$
The more training samples used to train the better the non-linear methods behave.
Note that in blue you can see the error that a linear method obtains which is obtained by setting 
$$c_i = 0, \; \forall i>n$$
thus the tendency is exactly the $\text{Var}(c_i)=\mathbb{E}_{u}(\| P_{v_i}u \|^2)\approx n^{-1}$.


In [4]:
n_linear = 3
n_to_learn = 1

In [11]:
@widgets.interact_manual(
    n=widgets.IntSlider(min=1, max=k_max, step=1, value=3, description=r"$\text{dim}(V_n)$"),
    n_non_linear=widgets.IntSlider(min=1, max=k_max, step=1, value=k_max, description=r"$\text{dim}(V_{nl})$"),
    n_train=widgets.FloatLogSlider(min=2, max=int(np.log10(N_train_max)), step=0.2, value=N_train_max, description="N training"),
    non_linear_crom_nn=False,
    non_linear_crom_rf=True,
    non_linear_crom_q=False,
    log_x=True,
    log_y=True,
)
def plot_learning(n=3, n_non_linear=25, n_train=N_train_max, non_linear_crom_nn=False, non_linear_crom_rf=False, non_linear_crom_q=False, log_x=True, log_y=True):
    models = [LINEAR_MODEL]
    if non_linear_crom_nn: models.append(NCROM_NN)
    if non_linear_crom_rf: models.append(NCROM_RF)
    if non_linear_crom_q: models.append(NCROM_Q)
    

    global n_to_learn
    global n_linear    
    global non_linear_models
    global errors
    global hashes
    smoothness = power = 1.  # this was meant to relax the manifold to be smoother than just a step function
    n_to_learn = n_non_linear
    n_linear = n
    n_train = int(n_train)
    
    # ----- create dataset for training non linear models ----- #
    coeffs_known_test, coeffs_to_learn_test = map(np.array, sample_space_for_non_linear_train(n_train=N_train_max, n=n, n_non_linear=n_non_linear,
                                                                      k_max=k_max, power=power,
                                                                      c_lim=c_lim, h_lim=h_lim, w_lim=w_lim, seed=43))
    valid_indexes = (np.isnan(coeffs_known_test).sum(axis=1)==0)*(np.isnan(coeffs_to_learn_test).sum(axis=1)==0)
    coeffs_known_test = coeffs_known_test[valid_indexes]
    coeffs_to_learn_test = coeffs_to_learn_test[valid_indexes]

    # this parameter is mostly to have a clear plot
    threshold = 1e-8  # if coefficients are small force zero and not learn
    # valid_columns_known = np.sqrt((coeffs_known**2).mean(axis=0)) > threshold
    valid_columns_to_learn = np.sqrt((coeffs_to_learn_test**2).mean(axis=0)) > threshold
    coeffs_to_learn_test = coeffs_to_learn_test[:, valid_columns_to_learn]

    coeffs_known, coeffs_to_learn = map(np.array, sample_space_for_non_linear_train(n_train=n_train, n=n, n_non_linear=n_non_linear,
                                                                      k_max=k_max, power=power,
                                                                      c_lim=c_lim, h_lim=h_lim, w_lim=w_lim))
    valid_indexes = (np.isnan(coeffs_known).sum(axis=1)==0)*(np.isnan(coeffs_to_learn).sum(axis=1)==0)
    coeffs_known = coeffs_known[valid_indexes]
    coeffs_to_learn = coeffs_to_learn[valid_indexes]
    coeffs_to_learn = coeffs_to_learn[:, valid_columns_to_learn]
    
    # -------- TRAIN -------- #
    error = dict()
    for model_name in non_linear_models:
        if model_name not in models: continue
        print("Training model", model_name)
        # non_linear_models[model_name], error, hashes = train(non_linear_models[model_name], n_train, n, n_non_linear, k_max, c_lim, h_lim, w_lim, error=None, hashes=0)
        non_linear_models[model_name].fit(np.array(coeffs_known), np.array(coeffs_to_learn))
        error[model_name] = (np.array(coeffs_to_learn_test) - non_linear_models[model_name].predict(
            np.array(coeffs_known_test)).reshape(coeffs_to_learn_test.shape)) ** 2
    clear_output(wait=True)

    # -------- PLOT -------- #
    plt.figure(figsize=(6, 6))
    x_ticks = np.arange(n, n + n_non_linear)[valid_columns_to_learn]

    # Correct f-string escaping: {{ and }} turn into literal { and }
    label = f"$k^{{-{power:.2f}}}$"
    # label=fr"$k^{{{-power:.2}}}$"
    plt.plot(x_ticks, x_ticks ** (-power), ":k", label=label)
    plt.plot(x_ticks, np.sqrt((np.array(coeffs_to_learn_test) ** 2).mean(axis=0)), ".-", label=LINEAR_MODEL)
    for model_name in tqdm(non_linear_models, desc="Training..."):
        if model_name not in models: continue
        plt.plot(x_ticks, np.sqrt(error[model_name].mean(axis=0)), ".-", label=model_name, c=colors_dict[model_name])
    plt.xticks(x_ticks[::max(1, len(x_ticks) // 10)])
    plt.ylabel("Prediction error")
    plt.xlabel("Unknown fourier coefficients")
    if log_x: plt.xscale("log")
    if log_y: plt.yscale("log")
    plt.legend(loc="lower center", bbox_to_anchor=(0.5, 1.05))
    # plt.tight_layout()


interactive(children=(IntSlider(value=3, description='$\\text{dim}(V_n)$', min=1), IntSlider(value=100, descri…

## Exploring non-linear manifolds

Here we can play with the definition of a manifold that initially is linear but we can further perturbe in a quadratic or sinusoidal way. You can also see the effect of adding noise and change the number of available points to sample the manifold.

In [12]:
@jax.jit
def sigmoid(x):
    return 1.0 / (1 + jnp.exp(-x))


@jax.jit
def linear_model(x, direction, position):
    return x[:, jnp.newaxis] * direction[jnp.newaxis, :] + position[jnp.newaxis, :]


@jax.jit
def map_perturbation_in_Vspace(y, orth_direction):
    return y[:, jnp.newaxis] * orth_direction[jnp.newaxis, :]


@jax.jit
def quadratic_perturbation(x, curvature):
    return curvature * x ** 2


@jax.jit
def sin_perturbation(x, f):
    return jnp.sin(2 * jnp.pi * f * x)


def get_manifold_sampler(orth_direction, direction, position, mlinear, mpoly, msin, c_quadratic, c_sin):
    @jax.jit
    def manifold(x):
        p = mlinear(x, direction, position)
        poly_perturbation = mpoly(x)
        p += c_quadratic * map_perturbation_in_Vspace(poly_perturbation, orth_direction)
        sin_perturbation = msin(x)
        p += c_sin * map_perturbation_in_Vspace(sin_perturbation, orth_direction)
        return p

    return manifold


@widgets.interact(n=widgets.IntSlider(min=1, max=1000, step=1, value=100),
                  theta=widgets.IntSlider(min=0, max=360, step=1, value=0),
                  c_quadratic=widgets.FloatSlider(min=0, max=1.0, step=0.05, value=0),
                  c_sin=widgets.FloatSlider(min=0, max=1.0, step=0.05, value=0),
                  frequency=widgets.FloatSlider(min=1.0, max=5.0, step=0.1, value=1.0),
                  noise=widgets.FloatSlider(min=0.01, max=.25, step=0.01, value=0),
                  pca_directions=False
                  )
def plot_manifold(n, theta, c_quadratic, c_sin, frequency, noise, pca_directions):
    theta = theta * jnp.pi / 180
    direction = jnp.array([jnp.cos(theta), jnp.sin(theta)])
    orth_direction = jnp.array([direction[1], -direction[0]])
    position = jnp.array([0., 0.])
    manifold = get_manifold_sampler(
        orth_direction=orth_direction,
        direction=direction,
        position=position,
        mlinear=linear_model,
        mpoly=partial(quadratic_perturbation, curvature=0.5),
        c_quadratic=c_quadratic,
        msin=partial(sin_perturbation, f=frequency),
        c_sin=c_sin
    )

    scale = 1.0
    x = jnp.array(np.random.normal(loc=0.0, scale=scale, size=(n,)))
    p = manifold(x) + noise * jnp.array(np.random.normal(size=(n, 2)))
    r = scale
    p = p[jnp.sum((p - position) ** 2, axis=1) <= r ** 2]

    # ----- plot ----- #
    # plt.figure(figsize=(8, 8))
    plt.scatter(*p.T, s=5)

    plt.xlim((-2 * scale + position[0], 2 * scale + position[0]))
    plt.ylim((-2 * scale + position[1], 2 * scale + position[1]))
    plt.axis("off")

    # ----- PCA ----- #
    if pca_directions:
        mean = jnp.mean(p, axis=0)
        (u, s, vh) = jnp.linalg.svd((p - mean).T @ (p - mean))
        pc1 = jnp.sign(direction @ vh[:, 0]) * vh[:, 0]
        pc2 = jnp.sign(orth_direction @ vh[:, 1]) * vh[:, 1]
        normalizer = jnp.sqrt(jnp.sum(s ** 2)) * 2
        pc2plot = (s[0] * pc1 / normalizer, s[1] * pc2 / normalizer)
        plt.arrow(*mean, *pc2plot[0])
        plt.text(*(mean + pc2plot[0] + 0.15 * pc2plot[0] / jnp.linalg.norm(pc2plot[0])), "PC1", ha='center',
                 va='center')
        plt.arrow(*mean, *pc2plot[1])
        plt.text(*(mean + pc2plot[1] + 0.15 * pc2plot[1] / jnp.linalg.norm(pc2plot[1])), "PC2", ha='center',
                 va='center')




interactive(children=(IntSlider(value=100, description='n', max=1000, min=1), IntSlider(value=0, description='…